# RAG — Retrieval-Augmented Generation

## 1. What is RAG?

> **RAG (Retrieval-Augmented Generation) is a technique that allows an LLM to retrieve relevant information from an external knowledge source and use that information to generate a better, context-aware answer.**

In simple words:

> **RAG = Retrieve relevant information → Give it to the LLM → Generate the answer**

Instead of asking the LLM to answer only from what it learned during training, we first provide it with relevant external information.

---

# 2. Why Do We Need RAG?

LLMs have some important limitations.

### Problem 1: Knowledge Cutoff

An LLM may not know information that appeared after its training data.

```text
New Information
      ↓
LLM may not know it
```

### Problem 2: Private Data

Suppose your company has:

```text
Company PDFs
Employee Policies
Product Documentation
Internal Wiki
Customer Data
```

The LLM normally doesn't know this private information.

### Problem 3: Hallucination

An LLM may sometimes generate information that sounds correct but isn't supported by the available facts.

RAG helps by providing relevant external context.

---

# 3. Basic RAG Architecture

The basic architecture is:

```text id="zq4r5n"
              KNOWLEDGE BASE
                    │
                    ▼
              Documents
                    │
                    ▼
             Document Loader
                    │
                    ▼
              Text Splitter
                    │
                    ▼
             Embedding Model
                    │
                    ▼
               Vector Store
                    │
                    │
                    ▼
                  Retriever
                    ▲
                    │
               User Query
                    │
                    ▼
            Relevant Documents
                    │
                    ▼
                  Prompt
                    │
                    ▼
                   LLM
                    │
                    ▼
                 Answer
```

This is the architecture you should remember for interviews.

---

# 4. RAG Has Two Main Phases

RAG can be divided into:

```text id="m2v1rc"
1. Indexing
2. Retrieval + Generation
```

---

# 5. Phase 1 — Indexing

Before users ask questions, we prepare the knowledge base.

Suppose we have:

```text id="f3g7vl"
company_handbook.pdf
```

### Step 1: Load Documents

```text id="c9zv0f"
PDF
 ↓
Document Loader
```

LangChain provides loaders for PDFs, websites, CSVs, text files, etc.

---

### Step 2: Split Documents

Large documents are divided into smaller chunks.

```text id="c5f8qa"
Large Document
      ↓
Text Splitter
      ↓
Chunk 1
Chunk 2
Chunk 3
Chunk 4
```

Why?

Because embedding and retrieval work better when we search meaningful pieces instead of an entire huge document.

---

### Step 3: Generate Embeddings

Each chunk is converted into a vector.

```text id="h8w0v5"
Chunk
 ↓
Embedding Model
 ↓
[0.12, -0.43, 0.87, ...]
```

---

### Step 4: Store Vectors

The vectors are stored in a vector store/database.

```text id="1ty4on"
Chunks
  +
Embeddings
  +
Metadata
       ↓
Vector Store
```

Examples:

```text id="n0r9y7"
Chroma
Qdrant
Pinecone
Weaviate
Milvus
pgvector
```

So the indexing pipeline is:

```text id="8x1d0b"
Documents
   ↓
Load
   ↓
Split
   ↓
Embed
   ↓
Store
```

---

# 6. Phase 2 — Retrieval + Generation

Now the user asks:

> "What is the company's leave policy?"

### Step 1: User Query

```text id="j43d3v"
"What is the company's leave policy?"
```

---

### Step 2: Convert Query into an Embedding

```text id="l2g7f1"
Query
 ↓
Embedding Model
 ↓
Query Vector
```

---

### Step 3: Retrieve Relevant Chunks

The query vector is compared with vectors in the vector store.

```text id="v6f8b4"
Query Vector
     ↓
Vector Store
     ↓
Similarity Search
     ↓
Top-K Relevant Chunks
```

For example:

```text id="8ap1vf"
Chunk 127 → Leave policy
Chunk 231 → Annual leave
Chunk 456 → Sick leave
```

---

### Step 4: Add Retrieved Context to Prompt

The retrieved information is given to the LLM.

```text id="2o8gwr"
System Instructions
       +
Retrieved Context
       +
User Question
       ↓
      LLM
```

---

### Step 5: Generate Answer

The LLM uses the retrieved context to generate the response.

```text id="0h7vhi"
Context + Question
        ↓
       LLM
        ↓
      Answer
```

---

# 7. Complete RAG Flow

Remember this:

```text id="c1k5bg"
                INDEXING
                   │
                   ▼
              Documents
                   ↓
              Loaders
                   ↓
             Text Splitter
                   ↓
            Embedding Model
                   ↓
             Vector Store
                   │
                   │
═══════════════════╪═══════════════════
                   │
                QUERY
                   │
                   ▼
              User Question
                   ↓
            Query Embedding
                   ↓
              Retriever
                   ↓
           Relevant Chunks
                   ↓
                 Prompt
                   ↓
                  LLM
                   ↓
                Answer
```

---

# 8. Why Is It Called "Retrieval-Augmented Generation"?

Let's break down the name.

### Retrieval

Find relevant information.

```text
Query
 ↓
Retrieve Documents
```

### Augmented

Add the retrieved information to the LLM's context.

```text
Question
+
Retrieved Context
```

### Generation

The LLM generates the final answer.

```text
Context + Question
       ↓
      LLM
       ↓
    Answer
```

Therefore:

> **Retrieval + Augmentation + Generation = RAG**

---

# 9. RAG vs Normal LLM

## Normal LLM

```text id="z5n1p3"
User Question
      ↓
      LLM
      ↓
    Answer
```

The LLM relies primarily on its existing learned parameters and whatever context you provide directly.

---

## RAG

```text id="m0b5xk"
User Question
      ↓
   Retriever
      ↓
External Knowledge
      ↓
    Context
      ↓
      LLM
      ↓
    Answer
```

The LLM gets additional relevant information at query time.

---

# 10. Simple Real-World Example

Imagine you build a chatbot for a company.

You have:

```text id="p5ujk3"
HR Policy.pdf
Leave Policy.pdf
Salary Policy.pdf
Work From Home.pdf
Insurance Policy.pdf
```

You create a RAG system.

Employee asks:

> "How many paid leaves can I take?"

RAG does:

```text id="u8k2sk"
Question
   ↓
Retriever
   ↓
Leave Policy.pdf
   ↓
Relevant Chunk
   ↓
LLM
   ↓
"According to the leave policy..."
```

The LLM doesn't need to memorize the company policy.

It retrieves it when needed.

---

# 11. RAG Does Not Train the LLM

This is a **very important interview point**.

Many beginners think:

> "When I put my PDF into RAG, the LLM learns the PDF."

That's incorrect.

RAG does **not** normally update the model's weights.

Instead:

```text id="l3k2wb"
PDF
 ↓
Embeddings
 ↓
Vector Store
```

Then at query time:

```text id="m4x8qp"
Query
 ↓
Retrieve PDF chunks
 ↓
Give chunks to LLM
 ↓
Generate answer
```

The model isn't being retrained.

### Interview Answer

> **RAG provides external knowledge to the LLM at inference time; it does not normally modify the LLM's parameters.**

---

# 12. RAG vs Fine-Tuning

Another very common interview question.

| RAG                                | Fine-Tuning                                |
| ---------------------------------- | ------------------------------------------ |
| Retrieves external information     | Trains model on additional examples        |
| Knowledge can be updated easily    | Updating requires another training process |
| Doesn't modify model weights       | Modifies model behavior/weights            |
| Good for private/current knowledge | Good for behavior/style/task adaptation    |
| Uses retrieval                     | Uses training                              |
| Common for document Q&A            | Common for specialized behavior            |

### Example

You have:

```text
10,000 company documents
```

You want the chatbot to answer questions based on those documents.

Usually:

> **RAG is a natural approach.**

If you want the model to consistently follow a particular output style or task format:

> **Fine-tuning may be appropriate.**

They can also be used together.

---

# 13. RAG Components

A typical RAG system contains:

### 1. Document Loaders

Load data.

```text
PDF
CSV
TXT
Websites
DOCX
Database
```

### 2. Text Splitters

Break documents into chunks.

### 3. Embedding Model

Convert chunks into vectors.

### 4. Vector Store

Store and search vectors.

### 5. Retriever

Retrieve relevant chunks.

### 6. Prompt

Combines context + question.

### 7. LLM

Generates the answer.

---

# 14. RAG With LangChain

A simplified LangChain architecture:

```text id="5y0f8q"
Document Loader
      ↓
Text Splitter
      ↓
Embeddings
      ↓
Vector Store
      ↓
Retriever
      ↓
Prompt
      ↓
LLM
      ↓
Answer
```

---

# 15. Simple LangChain RAG Example

Suppose you already have documents.

```python id="0yp6k7"
documents = [
    "RAG stands for Retrieval-Augmented Generation.",
    "RAG retrieves relevant information before generating an answer.",
    "Vector stores are used to search embedded documents."
]
```

Create embeddings:

```python id="8ytq8f"
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
```

Create vector store:

```python id="n9ex4h"
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="rag_demo",
    embedding_function=embeddings
)
```

Add documents:

```python id="s3f8tv"
vector_store.add_texts(documents)
```

---

# 16. Create Retriever

```python id="6g4m1k"
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)
```

Retrieve:

```python id="6q78qv"
docs = retriever.invoke(
    "What does RAG do?"
)
```

Now we have:

```text id="v7i4cv"
Question
   ↓
Retriever
   ↓
Relevant Documents
```

---

# 17. Add LLM

```python id="n9b7h5"
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini"
)
```

Create prompt:

```python id="91l2fv"
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context provided.

Context:
{context}

Question:
{question}
""")
```

---

# 18. Generate the Answer

```python id="7o1p3w"
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

response = llm.invoke(
    prompt.format_messages(
        context=context,
        question="What does RAG do?"
    )
)

print(response.content)
```

Architecture:

```text id="frb5tw"
Question
   ↓
Retriever
   ↓
Relevant Documents
   ↓
Context
   ↓
Prompt
   ↓
LLM
   ↓
Answer
```

---

# 19. Types of RAG

You will encounter different RAG architectures.

## Basic RAG

```text id="p8g5eq"
Query
 ↓
Retriever
 ↓
LLM
```

---

## Advanced RAG

```text id="t8n5wq"
Query
 ↓
Query Transformation
 ↓
Hybrid Retrieval
 ↓
Reranking
 ↓
Context Compression
 ↓
LLM
```

---

## Agentic RAG

An agent dynamically decides:

```text id="4c6k2m"
Question
   ↓
Agent
   ↓
Should I retrieve?
   ↓
Tool / Retriever
   ↓
Observe
   ↓
Reason
   ↓
Answer
```

This connects RAG with the **Agents** concept you're learning.

---

# 20. Common RAG Problems

RAG doesn't automatically guarantee correct answers.

### Problem 1: Bad Chunking

```text
Bad chunks
   ↓
Bad retrieval
   ↓
Bad context
   ↓
Bad answer
```

### Problem 2: Poor Embeddings

```text
Poor embeddings
   ↓
Wrong documents retrieved
```

### Problem 3: Wrong Top-K

Too few:

```text
Missing context
```

Too many:

```text
Noise + unnecessary tokens
```

### Problem 4: Poor Prompt

Even if retrieval is correct, a poor prompt can lead to a poor answer.

### Problem 5: Irrelevant Retrieval

If the Retriever returns incorrect information:

```text
Wrong Context
     ↓
LLM
     ↓
Potentially Wrong Answer
```

---

# 21. RAG Evaluation

A production RAG system should evaluate both:

### Retrieval

Did we retrieve the right information?

```text
Query
 ↓
Retriever
 ↓
Relevant?
```

### Generation

Did the LLM answer correctly using that information?

```text
Context
+
Question
 ↓
LLM
 ↓
Correct Answer?
```

So:

```text id="w4f6c8"
RAG Evaluation
      │
      ├── Retrieval Quality
      │
      └── Generation Quality
```

---

# 22. Interview Questions

## Beginner

### Q1. What is RAG?

> RAG stands for Retrieval-Augmented Generation. It retrieves relevant external information and provides it to an LLM as context before generating an answer.

### Q2. Why do we use RAG?

> To allow LLM applications to use external, private, domain-specific, or frequently changing information without retraining the model.

### Q3. Does RAG train the LLM?

> No. Standard RAG retrieves information at inference time and provides it as context; it does not normally update the model's weights.

---

# 23. Intermediate Questions

### Q4. What are the main components of RAG?

> Document loaders, text splitters, embedding models, vector stores, retrievers, prompts, and LLMs.

### Q5. What is the difference between RAG and fine-tuning?

> RAG provides external knowledge at inference time, while fine-tuning changes model behavior by training the model on additional data.

### Q6. Why do we split documents?

> To create manageable, semantically meaningful chunks that can be embedded and retrieved more effectively.

---

# 24. Scenario-Based Questions

### Q7. Your RAG system retrieves irrelevant documents. What would you investigate?

```text
Chunking
Embedding model
Query formulation
Top-K
Metadata filtering
Similarity metric
Hybrid retrieval
Reranking
```

### Q8. Your Retriever finds the correct document but the LLM gives a wrong answer. What could be wrong?

> The problem may be in context construction, prompt design, context length, LLM reasoning, or generation quality rather than retrieval.

---

# 25. 30-Second Revision

> **RAG = Retrieve + Augment + Generate**

```text
User Question
      ↓
Retriever
      ↓
Relevant Knowledge
      ↓
Prompt + Context
      ↓
LLM
      ↓
Answer
```

### Indexing:

```text
Documents
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector Store
```

### Query:

```text
Question
 ↓
Retrieve
 ↓
Context
 ↓
LLM
 ↓
Answer
```

### Remember:

> **RAG does not retrain the LLM. It gives the LLM relevant external context at inference time.**

---

# 26. 2-Minute Revision

## RAG

**Retrieval-Augmented Generation** is an architecture that combines information retrieval with LLM generation.

### Indexing Pipeline

```text
Documents
 ↓
Document Loader
 ↓
Text Splitter
 ↓
Embedding Model
 ↓
Vector Store
```

### Query Pipeline

```text
User Query
 ↓
Query Embedding
 ↓
Retriever
 ↓
Relevant Chunks
 ↓
Prompt
 ↓
LLM
 ↓
Answer
```

### Why RAG?

* Private/company knowledge
* Current or frequently changing information
* Domain-specific documents
* Reducing unsupported answers
* Grounding LLM responses in retrieved context

### RAG vs Fine-Tuning

```text
RAG
→ External knowledge
→ Inference-time retrieval
→ No weight updates

Fine-Tuning
→ Model behavior/task adaptation
→ Training
→ Model weights are updated
```

### Golden Interview Answer

> **RAG is a technique where relevant information is retrieved from an external knowledge source and added to an LLM's context before generation. A typical RAG system loads and chunks documents, converts them into embeddings, stores them in a vector store, retrieves relevant chunks for a user query, and passes those chunks to an LLM to generate a grounded response.**
